### ガウス混合線形モデルの作成と回帰性能の評価



単一線形回帰回帰とクラスタリングの知見を利用した区分線形回帰を行い比較します。
以下の５つを行い比較します。

1. descriptor $x$
2. $x+x^2$
3. $x*$ cluster pribability
4. $(x+x2)*$ cluster pribability



結晶、結晶内の原子、各原子の説明変数が三次元データとして用意されています。

df(crystal, atom, desrcriptor)

線形回帰の場合は結晶説明変数は原子説明変数の和として表されます。
結晶説明変数に対して全エネルギーの回帰を行います。

In [ ]:
from sklearn.linear_model import RidgeCV, Ridge
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import sklearn.preprocessing
import seaborn as sns
import pandas as pd
import os
import numpy as np
import matplotlib.pylab as plt
%matplotlib inline

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 60)

warnings.filterwarnings('ignore')


In [ ]:
g_df_clusterprobability = pd.read_csv(os.path.join(
    "..", "data_calculated", "Carbon8_yproba.csv"), index_col=[0, 1])


In [ ]:
def load_energy():
    """load data from files.

    Returns:
        pd.DataFrame: data.
    """
    df_energy = pd.read_csv(os.path.join(
        "..", "data", "Carbon8_Etot_spin.csv"), index_col=[0])
    energy_min = np.min(df_energy.values)
    df_energy["Etot"] = (df_energy.values-energy_min)/8
    return df_energy


g_df_energy = load_energy()


In [ ]:
g_y = g_df_energy["Etot"].values
g_df_energy.plot(y="Etot")


すでに計算した原子説明変数を読み込みます。

In [ ]:
g_df_atomX1 = pd.read_csv(os.path.join(
    "..", "data_calculated", "Carbon8_descriptor.csv"), index_col=[0, 1])


indexが同じ(同じ並びである)ことを確認しておきます。

In [ ]:
np.all(g_df_atomX1.index.levels[0] == g_df_energy.index)


何も操作しないcell説明変数に対してRidge回帰を行います。

結果保存用の辞書変数scoresを定義しておきます。

In [ ]:
g_scores = {}


まず、変数Xとyを作成します。

In [ ]:
def make_cell_descriptor(df_atom):
    """make cell desccriptor from atom descriptor

    Args:
        df_atom (pd.DataFrame): data

    Returns:
        np.array: normalized cell descriptor
    """
    ncryst = len(df_atom.index.levels[0])
    natom = 8
    natomdesc = df_atom.shape[1]
    Xatom = df_atom.values.reshape((ncryst, natom, natomdesc))
    Xcell = np.sum(Xatom, axis=1)
    scaler = StandardScaler()
    X = scaler.fit_transform(Xcell)
    return X


g_X = make_cell_descriptor(g_df_atomX1)


Ridge回帰を行う。RidgeCVを用いてscoreを計算します。

In [ ]:
def evaluate_cv_score(X, y, n_split=5):
    """evaluate CV score

    Args:
        X (np.array): descriptor
        y (np.array): target values
        n_split (int, optional): the number of splits in CV. Defaults to 5.

    Returns:
        float: R2 score
    """
    kf = KFold(n_split, shuffle=True)
    reg = RidgeCV(cv=kf, fit_intercept=True)
    reg.fit(X, y)
    score = reg.score(X, y)
    return score


g_score = evaluate_cv_score(g_X, g_y)
g_scores["X1"] = [g_score]
print(g_score)


In [ ]:
import copy


def save_Xy(X, y, index, columns_X, columns_y):
    """save X and y

    Args:
        X (np.array): descriptor
        y (np.array): target values
        index (list): index of samples
        columns_X (list): a list of feature names
        columns_y (list): target name
    """
    filename = "../data_calculated/Carbon8_cell_descriptor_Etot.csv"
    if not os.path.exists(filename):
        columns = copy.copy(list(columns_X))
        columns.append(columns_y)
        df = pd.DataFrame(
            np.hstack([X, y.reshape(-1, 1)]), index=index, columns=columns)
        df.to_csv(filename)

    # 保存内容のの確認
    dftmp = pd.read_csv("../data_calculated/Carbon8_cell_descriptor_Etot.csv")
    display(dftmp)

# 全探索用に保存する。
# save_Xy(X,y,df_energy.index,df_atomX1.columns,df_energy.columns[0])


次にGMMにより計算されたクラスター確率をかけて説明変数を作成する。

In [ ]:
def make_cellprobability_descriptor(df_atom, df_clusterprobability):
    """make cell descriptor
        = sum_atom atomic_descriptor(atom) * cluster_probability(atom)

    Args:
        df_atom (pd.DataFrame): atom descriptor
        df_clusterprobability (pd.DataFrame): cluster probability

    Returns:
        np.array: cell descriptor
    """
    ncryst = len(df_atom.index.levels[0])
    natom = 8
    Xproba = np.zeros((df_atom.shape[0],
                       df_atom.shape[1] * df_clusterprobability.shape[1]))
    for i in range(df_atom.shape[0]):
        x = df_atom.iloc[i, :].values
        proba = df_clusterprobability.iloc[i, :].values
        xproba = x.T.reshape(-1, 1)*proba
        Xproba[i, :] = xproba.reshape(-1)
    Xproba = Xproba.reshape(ncryst, natom, -1)
    scaler = StandardScaler()
    Xcell = np.sum(Xproba, axis=1)
    X = scaler.fit_transform(Xcell)
    return X


In [ ]:
g_X = make_cellprobability_descriptor(g_df_atomX1, g_df_clusterprobability)
g_score = evaluate_cv_score(g_X, g_y)
g_scores["X*P"] = [g_score]
print(g_score)


$X , X^2$(対角項のみ）で説明変数を作成する。

In [ ]:
def make_atomX2(df_atom):
    """make descriptor made of X and X2

    Args:
        df_atom (pd.DataFrame): atom desriptor

    Returns:
        pd.DataFrame: X and X2 data
    """
    X1 = df_atom.values
    X2 = X1*X1
    Xraw = np.hstack([X1, X2])
    dfatom = pd.DataFrame(Xraw, index=df_atom.index)
    return dfatom


In [ ]:
g_dfatomX2 = make_atomX2(g_df_atomX1)
g_X = make_cell_descriptor(g_dfatomX2)
g_y = g_df_energy["Etot"].values
g_score = evaluate_cv_score(g_X, g_y)
g_scores["(X,X2)"] = [g_score]
print(g_score)


$X , X^2$説明変数にクラスタ確率を掛けて説明変数を作成してみる。

In [ ]:
g_X = make_cellprobability_descriptor(g_dfatomX2, g_df_clusterprobability)
g_y = g_df_energy["Etot"].values
g_score = evaluate_cv_score(g_X, g_y)
g_scores["(X,X2)*P"] = [g_score]
print(g_score)


In [ ]:
g_df_result = pd.DataFrame(g_scores, index=["score"])
g_df_result.T.sort_values(by="score").plot.bar(ylim=(0.8, 1))
plt.ylabel("$R^2$")
if not os.path.isdir("image_executed"):
    os.makedirs("image_executed")
plt.savefig("image_executed/carbon8_result.png")

結晶説明変数に対して区分線形モデルを作成します。
まず、結晶説明変数に対してクラスタリングしてみます。

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap,TSNE


def show_cell_descriptor_cluster(X):
    """dimensional reduction on cell descriptor with isomap
        show them in 2D.

    Args:
        X (np.array): descriptor
    """
    # dr = PCA(2)
    # dr = Isomap(n_components=2)
    dr = TSNE(2)
    Xt = dr.fit_transform(X)
    
    plt.figure()
    plt.plot(Xt[:, 0], Xt[:, 1], ".", alpha=0.7)
    plt.xlabel("feature1")
    plt.xlabel("feature2")
    plt.show()


show_cell_descriptor_cluster(g_X)


残念ながらこの結晶説明変数に対してクラスタリングができるようには見えません。
結晶説明変数のクラス分けを原子説明変数クラスの平均とします。

In [ ]:
def evaluate_cv_ytestp(X, y, n_splits=5):
    """evaluate y_test^predict

    Args:
        X (np.array): descriptor
        y (np.array): target values
        n_splits (int, optional): the number of splits in CV. Defaults to 5.

    Returns:
        np.array: a list of y_test
        np.array: a list of y_test^predict
    """
    kfcv = KFold(n_splits, shuffle=True)
    regcv = RidgeCV(cv=kfcv, fit_intercept=True)
    regcv.fit(X, y)
    print("CV alpha=", regcv.alpha_)
    reg = Ridge(alpha=regcv.alpha_)
    kf = KFold(n_splits, shuffle=True)
    ytest_list = []
    ytestp_list = []
    for train, test in kf.split(X):
        Xtrain, ytrain = X[train], y[train]
        Xtest, ytest = X[test], y[test]
        reg.fit(Xtrain, ytrain)
        ytestp = reg.predict(Xtest)
        ytest_list.extend(ytest)
        ytestp_list.extend(ytestp)
    return ytest_list, ytestp_list


In [ ]:
def make_cell_clusterid(df_atomX1, df_clusterprobability):
    """make cell cluster descriptor

    Args:
        df_atomX1 (pd.DataFrame): descriptor
        df_clusterprobability (pd.DataFrame): cluster probability

    Returns:
        pd.DataFrame: cluster probability to multiply atom descriptor
    """
    ncryst = len(df_atomX1.index.levels[0])
    natom = 8
    atom_proba = df_clusterprobability.values.reshape(ncryst, natom, -1)
    cell_proba = np.sum(atom_proba, axis=1)
    df_cell_proba = pd.DataFrame(cell_proba, index=df_atomX1.index.levels[0])
    df_cell_clusterid = pd.DataFrame(np.argmax(df_cell_proba.values, axis=1),
                                     index=df_atomX1.index.levels[0], columns=["cluster"])
    return df_cell_clusterid


g_df_cell_clusterid = make_cell_clusterid(g_df_atomX1, g_df_clusterprobability)


In [ ]:
from collections import Counter
g_clist = Counter(g_df_cell_clusterid.values.reshape(-1).tolist())
print(g_clist)


In [ ]:
def evaluate_cv_cluster_ytestp(df_atomX1, df_cell_clusterid, df_energy, clist):
    """evaluate y_test^p

    Args:
        df_atomX1 (pd.DataFrame): atomic descriptor
        df_cell_clusterid (pd.DataFrame): atomic cluster probability 
        df_energy (pd.DataFrame): target values.
        clist (Counter): Cs and their occurrence.

    Returns:
        np.array: y_test^observed.
        np.array: y_test^predict.
    """
    X1 = make_cell_descriptor(df_atomX1)
    y1 = df_energy["Etot"].values
    ytest_list = []
    ytestp_list = []
    for c in clist.keys():
        selected = df_cell_clusterid.values == c
        selected = selected.reshape(-1)
        X = X1[selected, :]
        y = y1[selected]
        ytest, ytestp = evaluate_cv_ytestp(X, y)
        ytest_list.extend(ytest)
        ytestp_list.extend(ytestp)
    ytest = np.array(ytest_list)
    ytestp = np.array(ytestp_list)
    return ytest, ytestp


g_ytest, g_ytestp = evaluate_cv_cluster_ytestp(
    g_df_atomX1, g_df_cell_clusterid, g_df_energy, g_clist)
g_score = r2_score(g_ytest, g_ytestp)
g_scores["partitioned"] = g_score
print("score=", g_score)


In [ ]:
def show_cluster_reg_scores(df_atomX1, df_energy, df_cell_clusterid, clist):
    X1 = make_cell_descriptor(df_atomX1)
    y1 = df_energy["Etot"].values
    result = []
    for c in clist.keys():
        selected = df_cell_clusterid.values == c
        selected = selected.reshape(-1)
        X = X1[selected, :]
        y = y1[selected]
        score = evaluate_cv_score(X, y)
        result.append([c, score])
    return pd.DataFrame(result, columns=["cluster#", "R2"]).sort_values(by="cluster#").set_index("cluster#")


show_cluster_reg_scores(g_df_atomX1, g_df_energy, g_df_cell_clusterid, g_clist)


最後に、ここで示した$R^2$はクロスバリデーションでのテストデータに対する評価値では無いことに注意してください。


#### 参考文献

1. Tien Lam Pham, Hiori Kino, Kiyoyuki Terakura, Takashi Miyake, and Hieu Chi Dam,
"Novel mixture model for the representation of potential energy surfaces",
The Journal of Chemical Physics 145, 154103 (2016).

